<a href="https://colab.research.google.com/github/Fespinoza1477/ds2002-fa26/blob/main/notebooks/04-json-apis/2026-10-02%20%E2%80%94%20Normalize%20and%20Join%20Nested%20JSON%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [ ]:
import pandas as pd
import numpy as np
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-01','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** Line Item



### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [ ]:
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order', 'vendor_id'])
assert len(lines) == 4, "Expected 4 rows"
assert lines['qty'].sum() == 7, "Expected quantity sum to be 7"

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [ ]:
vendors_df = pd.DataFrame(vendors_json)
joined = lines.merge(vendors_df, on='vendor_id', how='left', indicator=True)
unmatched = joined[joined['_merge'] == 'left_only']
if unmatched.empty:
  print("Q2: No unmatched vendors in this data fixture (despite the prompt's warning).")
else:
  print(f"Q2: Unmatched vendor(s): {unmatched['vendor_id'].unique()}")
  print(f"Q2: Units attached to unmatched vendor: {unmatched['qty'].sum()}")


### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [ ]:
vendor_totals = joined.groupby('name', dropna=False)['qty'].sum()
print("\nQ3: Quantity per vendor name:\n", vendor_totals)

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [ ]:
zone_totals = joined.groupby('zone', dropna=False)['qty'].sum()
print('\nQ4: Zone totals:\n', zone_totals)
print('The total across all zones is 7. If there were an unmatched vendor with no zone, those units would appear under NaN zone group.')

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [ ]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                                # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]
filtered_json = [order for order in ragged_json if 'lines' in order]
ragged_lines = pd.json_normalize(filtered_json, record_path='lines', meta=['order','vendor_id'])
ragged_lines['qty'] = pd.to_numeric(ragged_lines['qty'], errors='coerce')
print("\nQ5: Ragged payload normalized:\n", ragged_lines)

### Q6 — Validate

In [ ]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a). "One row per line item" is the useful shape because it allows you to directly group and aggregate quantities by item, vendor, or zone. If you kept the data as one row per order, the individual items and qunatities would remain trapped in nested lists or dictionaries, making it impossible to perform standard pandas vector operations like .groupby() and .sum() without writing custom loop functions for every row.

b)A missing quantity (NaN) indicated that the data was not recorded, transmitted, or captured properly; it is an unknown value. A quantity of zero(0) is a factual, explicit measurement of nothing. If you conflate them by filing NaNs with 0, you artificially lower overall averages, falsify the order history, and mask underlying data collection errors from upstream systems.